# 07_ex7_effort_estimate.sql  Exercise 7：業務効果を確認する（10分）

注意 : 以下の時間はすべて「PoC で測定すべき仮説」です。確定した効果ではありません。  
現状の時間（合計 100 時間）も説明用の架空の値です。実際の内訳は業務担当者へのヒアリングや PoC の実測で置き換えてください。  

> **💡 解説**
> - **なぜ**：効果を断定せず、工程ごとの仮説として置き、PoC で何を測るかを決めます。

In [ ]:
%run ./00_config

このノートブックの SQL は `run_sql()`（`00_config` で定義）で実行します。テーブル・View・関数の名前には、`00_config` のカタログ・スキーマが自動で付きます（例：`sales_actual` → `workspace.vehicle_alias_handson.sales_actual`）。**最初に `%run ./00_config` のセルを実行してください。**

> **💡 解説**
> - **仕組み**：工程・現状の時間・導入後の想定・手段・PoC で測ることを、1行ずつ持つ表です。

In [ ]:
run_sql(r"""
CREATE OR REPLACE TABLE effort_estimate (
  task_order        INT,
  task              STRING  COMMENT '作業工程',
  current_hours     DOUBLE  COMMENT '現状の想定時間（新機種1台分のコスト積み上げ、仮置き）',
  target_hours      DOUBLE  COMMENT 'Databricks 導入後の想定時間（仮説。参加者が入力）',
  databricks_lever  STRING  COMMENT '短縮の手段',
  poc_measurement   STRING  COMMENT 'PoC で何を測るか'
) COMMENT '業務効果試算（仮説）。PoC で実測して検証する。';
""")

> **💡 解説**
> - **仕組み**：時間はすべて説明用の架空の値です。

In [ ]:
run_sql(r"""
INSERT INTO effort_estimate VALUES
 (1,'データ収集'        ,22, 8,'計画・販売・生産を Delta に集約（元システムは変更しない）'          ,'データ取得〜集計可能状態までの時間'),
 (2,'名称・コード調査'  ,20, 5,'Discover Page で正式定義・別名を参照、Genie One で質問'              ,'「このコードは何の車両か」の調査1件あたり時間・件数'),
 (3,'マッチング'        ,22, 7,'vehicle_alias_master と正規化ルールで自動変換、残りだけレビュー'    ,'自動変換率、レビュー対象件数、1件あたり確認時間'),
 (4,'不整合確認'        ,17, 4,'DQ ルール（タイヤ本数、重複、マスタ矛盾、欠損）で自動検出'          ,'検出件数、見逃し件数、原因調査時間'),
 (5,'集計'              ,11, 2,'共通機種ID × 月の採算 View で再集計（再実行は数秒）'                ,'再集計1回あたりの時間、再集計回数'),
 (6,'レビュー'          , 8, 6,'根拠（Page・対応表の承認記録）を添えたレビュー。判断自体は人が行う','レビュー時間、差し戻し件数');
""")

## 7-1. 仮説値で試算

> **💡 解説**
> - **仕組み**：工程ごとの行に、合計の行を `UNION ALL` で足しています。

In [ ]:
run_sql(r"""
SELECT task, current_hours, target_hours,
       current_hours - target_hours                                  AS saved_hours,
       round((current_hours - target_hours) / current_hours * 100, 1) AS saved_pct,
       databricks_lever, poc_measurement
FROM effort_estimate
UNION ALL
SELECT '合計', sum(current_hours), sum(target_hours),
       sum(current_hours) - sum(target_hours),
       round((sum(current_hours) - sum(target_hours)) / sum(current_hours) * 100, 1),
       NULL, NULL
FROM effort_estimate;
-- 期待値（仮説値のまま）: 100h → 32h、削減 68h、削減率 68.0%（すべて架空の値）
""")

## 7-2. 参加者が自分の想定値を入力して試算する（SQL エディタのパラメータ :h_xxx に数値を入力）

> **💡 解説**
> - **なぜ**：参加者が自分の想定値を入れて、削減の見込みを試算します。
> - **仕組み**：`:h_collect` のような名前付きパラメータを使います。ノートブックでは、ウィジェットの値を `run_sql(..., args=...)` で渡します。値は文字列なので `CAST(... AS DOUBLE)` で数値にします。ウィンドウ関数 `sum(...) OVER ()` で、行ごとの値と合計を同時に出しています。
> - **利点**：SQL を書き換えずに、条件だけを変えて何度でも試せます。

In [ ]:
# Exercise 7-2 用のパラメータ（ノートブック上部のウィジェットに自分の想定時間を入力）
for name, default in [("h_collect", "8"), ("h_research", "5"), ("h_matching", "7"),
                      ("h_quality", "4"), ("h_aggregate", "2"), ("h_review", "6")]:
    dbutils.widgets.text(name, default)

上のセルを実行すると、ノートブックの上部に入力欄（`h_collect` など）が表示されます。自分の想定時間を入力してから、次のセルを実行します。

> **💡 解説**
> - **仕組み**：`dbutils.widgets.text` はノートブックに入力欄（ウィジェット）を作ります。次のセルでは、その値を `args` として SQL の名前付きパラメータ（`:h_collect` など）に渡します。

In [ ]:
run_sql(r"""
WITH input AS (
  SELECT * FROM VALUES
    ('データ収集'      , CAST(:h_collect   AS DOUBLE)),
    ('名称・コード調査', CAST(:h_research  AS DOUBLE)),
    ('マッチング'      , CAST(:h_matching  AS DOUBLE)),
    ('不整合確認'      , CAST(:h_quality   AS DOUBLE)),
    ('集計'            , CAST(:h_aggregate AS DOUBLE)),
    ('レビュー'        , CAST(:h_review    AS DOUBLE)) AS t(task, my_target_hours)
)
SELECT e.task_order, e.task, e.current_hours, i.my_target_hours,
       e.current_hours - i.my_target_hours                                  AS saved_hours,
       round((e.current_hours - i.my_target_hours) / e.current_hours * 100, 1) AS saved_pct,
       sum(e.current_hours - i.my_target_hours) OVER ()                     AS total_saved_hours,
       round(sum(e.current_hours - i.my_target_hours) OVER () / sum(e.current_hours) OVER () * 100, 1) AS total_saved_pct
FROM effort_estimate e JOIN input i USING (task)
ORDER BY e.task_order;
""", args={k: dbutils.widgets.get(k) for k in ('h_collect', 'h_research', 'h_matching', 'h_quality', 'h_aggregate', 'h_review')})

## 7-3. （パラメータが使えない場合の代替）UPDATE で自分の想定値を入れてから 7-1 を再実行

```sql
UPDATE effort_estimate SET target_hours = 20 WHERE task = 'データ収集';
```